# Suite CCS — C and C++ simplified by ISO/IEC 9899 and ISO/IEC 14882, under their holder's ABI

`C.Simplifiers.OfAny(term, scope)` and `Cpp.Simplifiers.OfAny(term, scope)` simplify a term of the C family by its
language's standard, every kind the evaluation inventory gives a value folding: what the scope decides folds into the
literal of its value, as the language writes it (a suffix by type, `nullptr`, a floating value a binary64 literal spells
exactly), a value of a type no literal has into a cast of a literal (`(short)3`), and, in C, an array into a compound
literal (`(int[]){1, 2}`), each a closed constructor expression; the rest stays, the residual, its subterms simplified. The types it computes in are the stored domains its
holder reads, its own dialect's and its ABI's (`abi`, x86-64's System V ABI unless a case names AAPCS64's), so `sizeof`
and `_Generic` read a type's layout and identity there. What has no rule stays too, named with its class in the
evaluation inventory (`residues`): the object model and declarations (`d`), an operand's declared type (`c`), a layout
no ABI dialect stores (`a`), an initializer alone and a type name (`b`). What the standard makes an error still raises.
Terms are written through the dialects' writers and builders, in a transaction (`term`); `text` gives a residual's
source, as the printer writes it, and `rule` the folding rules' value of a term whose value has no literal.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as AAPCS64 from "@mbse/expressions/Arm/AAPCS64";
import * as CDialect from "@mbse/expressions/C/Expressions";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor";
import * as CTypes from "@mbse/expressions/C/Types";
import * as CppDialect from "@mbse/expressions/Cpp/Expressions";
import * as CppRegExp from "@mbse/expressions/Cpp/RegExp";
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as PosixBRE from "@mbse/expressions/POSIX/BRE";
import * as PosixERE from "@mbse/expressions/POSIX/ERE";
import * as AMD64 from "@mbse/expressions/SystemV/AMD64";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import { Errors as FE, Terms as F } from "@mbse/expressions/Framework";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
import { Builders, Errors } from "@mbse/schemas/Framework";
import { assert, equal, raises, same } from "./support.js";

const { NotEvaluated } = Simplification;
const { OverflowError, ZeroDivisionError } = FE;
const { ValueError } = Errors;
const [C, CV, CText] = [CDialect.Expressions, CDialect.Simplifiers, CDialect.Text];
const [X, XV, XText] = [CppDialect.Expressions, CppDialect.Simplifiers, CppDialect.Text];
type Dialect = typeof C | typeof X;

const store: any = new Builders.Store();
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install,
  C.install, X.install, AMD64.install, AAPCS64.install, CTypes.install, CppTypes.install, CPreprocessor.install,
  CppRegExp.install, PosixBRE.install, PosixERE.install]) {
  store.transaction(install);
}
for (const define of [C.define, X.define, AMD64.define, AAPCS64.define]) store.transaction(define); // the types C, C++ and each ABI define

/** The System V AMD64 ABI, read within the transaction open: the default ABI the cases read their terms under. */
function amd64(): any {
  return store.Instances.Expressions.SystemV.AMD64;
}

/** Arm's AAPCS64 ABI, read within the transaction open. */
function aapcs64(): any {
  return store.Instances.Expressions.Arm.AAPCS64;
}

/** The term `spec` writes in the dialect `of` (C or C++), a native value as its literal, the root of a holder built
 * within the transaction open that names the ABI its terms are read under (none: AMD64's); the term, a visitor of
 * that transaction, lives within it. */
function term(spec: unknown, of: Dialect = C, abi: unknown = undefined): any {
  abi = abi === undefined ? amd64() : abi;
  return F.term(store.Expressions.Framework.Expression((e: any) => e.root(of.DIALECT.spec(spec)).abi([abi])));
}

/** The simplification of the term `spec` writes, in `scope`. */
function simplified(spec: unknown, scope: any = {}, of: Dialect = C, abi: unknown = undefined): Simplification.Simplified {
  return (of === C ? CV : XV).OfAny(term(spec, of, abi), scope);
}

/** The source of the residual of the term `spec` writes, simplified in `scope`. */
function text(spec: unknown, scope: any = {}, of: Dialect = C, abi: unknown = undefined): string {
  const residual = simplified(spec, scope, of, abi).residual;
  abi = abi === undefined ? amd64() : abi;
  const holder = store.Expressions.Framework.Expression((e: any) => e.root(residual).abi([abi]));
  return (of === C ? CText : XText).ToText(holder);
}

/** The value the term `spec` writes simplifies to, in `scope`: its literal root's. */
const value = (spec: unknown, scope: any = {}, of: Dialect = C, abi: unknown = undefined): any => simplified(spec, scope, of, abi).evaluate();

/** The folding rules' value of the term `spec` writes, a `Typed`: where no literal spells it. */
function rule(spec: unknown, scope: any = {}, of: Dialect = C, abi: unknown = undefined): any {
  const rules = of === C ? CV : XV;
  return rules.RULES.evaluate(term(spec, of, abi), scope instanceof rules.Scope ? scope : new rules.Scope(scope));
}

/** Why what stayed in the residual stayed: each residue's reason and class. */
const reasons = (spec: unknown, scope: any = {}, of: Dialect = C, abi: unknown = undefined): [string, string][] =>
  simplified(spec, scope, of, abi).residues.map((residue) => [residue.reason, residue.category]);

// The builders of the terms the cases write: a type name, of its keywords or a type's name, and an abstract declarator
type Declarator = (d: any) => any;
function keywords(...words: (string | Declarator)[]): any {
  const declarator = typeof words[words.length - 1] === "function" ? words.pop() as Declarator : null;
  const specifiers = (words as string[]).map((w) => ["const", "volatile"].includes(w)
    ? (s: any) => s.cvqualifier((q: any) => q.keyword(w)) : (s: any) => s.primitivetypespecifier((p: any) => p.keyword(w)));
  return (u: any) => u.typeid((i: any) => declarator === null ? i.specifiers(specifiers) : i.specifiers(specifiers)
    .declarator(declarator));
}

function named(name: string): any {
  return (u: any) => u.typeid((i: any) => i.specifiers([(s: any) => s.namedtypespecifier(
    (n: any) => n.name((m: any) => m.identifier((d: any) => d.spelling(name))))]));
}

/** `_BitInt(width)`, or `unsigned _BitInt(width)`. */
function bits(width: bigint, unsigned = false): any {
  const sign = unsigned ? [(s: any) => s.primitivetypespecifier((p: any) => p.keyword("unsigned"))] : [];
  return (u: any) => u.typeid((i: any) => i.specifiers([...sign, (s: any) => s.bitintspecifier((b: any) => b.width(C.literal(width)))]));
}

/** An array declarator, of `size` (none: its initializer's), around `inner`. */
function array(size: unknown = null, inner: Declarator | null = null): Declarator {
  return (d: any) => d.arraydeclarator((r: any) => {
    const sized = size === null ? r : r.size(C.DIALECT.spec(size));
    return inner === null ? sized : sized.declarator(inner);
  });
}

const pointer: Declarator = (d) => d.pointerdeclarator((p: any) => p);
const [x, y] = [C.identifier("x"), C.identifier("y")];
const nullptr = (u: any) => u.constant((r: any) => r.singleton("nullptr"));

const paren = (spec: unknown, of: Dialect = C) => (u: any) => u.parenthesizedexpression((r: any) => r.expression(of.DIALECT.spec(spec)));
const extension = (spec: unknown) => (u: any) => u.extensionexpression((r: any) => r.operand(C.DIALECT.spec(spec)));
const string = (held: string, prefix: string | null = null) => (u: any) => u.literal((r: any) => prefix === null
  ? r.value((v: any) => v.as_scalar(held)) : r.value((v: any) => v.as_scalar(held)).prefix(prefix));
const concat = (...parts: unknown[]) => (u: any) => u.concatenatedstring((r: any) => r.parts(parts.map((part) => C.DIALECT.spec(part))));

/** GNU's `condition ?: alternative`. */
const gnu = (condition: unknown, alternative: unknown) => (u: any) => u.conditionalexpression((r: any) => r
  .condition(C.DIALECT.spec(condition)).alternative(C.DIALECT.spec(alternative)));

const sizeof = (operand: unknown, of: Dialect = C) => (u: any) => u.sizeofexpression((r: any) => r.operand(of.DIALECT.spec(operand)));
const alignof = (operand: unknown) => (u: any) => u.alignofexpression((r: any) => r.keyword("_Alignof").operand(C.DIALECT.spec(operand)));

/** `_Generic(controlling, type: value, ..., default: value)`, `default`'s type null. */
function generic(controlling: unknown, ...associations: [unknown, unknown][]): any {
  return (u: any) => u.genericselection((r: any) => r.controlling(C.DIALECT.spec(controlling)).associations(associations.map(
    ([t, v]) => (a: any) => t === null ? a.value(C.DIALECT.spec(v)) : a.type(t).value(C.DIALECT.spec(v)))));
}

/** `(type){items}`. */
const compound = (type: unknown, ...items: unknown[]) => (u: any) => u.compoundliteralexpression((r: any) => r.type(type)
  .initializer((i: any) => i.items(items.map((item) => C.DIALECT.spec(item)))));

/** `[index] = item`, or GNU's `[index ... last] = item`. */
function designated(index: bigint, item: unknown, last: bigint | null = null): any {
  return (u: any) => u.designatedinitializer((r: any) => r.designators([(d: any) => d.indexdesignator(
    (i: any) => last === null ? i.index(C.literal(index)) : i.index(C.literal(index)).last(C.literal(last)))])
    .initializer((i: any) => i.equalinitializer((e: any) => e.value(C.DIALECT.spec(item)))));
}

// C++'s
const ident = (name: string) => X.identifier(name);

function qualified(...names: string[]): any {
  return (u: any) => u.idexpression((r: any) => r.name((n: any) => n.qualifiedname((q: any) => q.qualifiers(
    names.slice(0, -1).map((s) => (m: any) => m.identifier((i: any) => i.spelling(s)))).name(
    (m: any) => m.identifier((i: any) => i.spelling(names[names.length - 1]))))));
}

const this_ = (u: any) => u.thisexpression((r: any) => r);
const udl = (held: unknown, suffix: string) => (u: any) => u.userdefinedliteral((r: any) => r.literal(
  (l: any) => l.value((v: any) => v.as_scalar(held))).suffix(suffix));

/** `keyword(args)`, or `keyword{args}`, a functional cast; a keyword C++ does not write names a type. */
function functional(keyword: string, args: unknown[] = [], braced = false): any {
  const specs = args.map((argument) => X.DIALECT.spec(argument));
  const specifier = ["int", "unsigned", "double", "bool", "char", "void"].includes(keyword)
    ? (s: any) => s.primitivetypespecifier((p: any) => p.keyword(keyword))
    : (s: any) => s.namedtypespecifier((n: any) => n.name((m: any) => m.identifier((i: any) => i.spelling(keyword))));
  const initializer = braced ? (i: any) => i.initializerlist((l: any) => l.items(specs))
    : (i: any) => i.parenthesizedinitializer((p: any) => p.arguments(specs));
  return (u: any) => u.functionalcastexpression((r: any) => r.type(specifier).initializer(initializer));
}

const static_cast = (type: unknown, operand: unknown, operator = "static_cast") => (u: any) => u.namedcastexpression(
  (r: any) => r.operator(operator).type(type).operand(X.DIALECT.spec(operand)));
const sizeof_pack = (name: string) => (u: any) => u.sizeofpackexpression((r: any) => r.pack((i: any) => i.spelling(name)));
const expansion = (pattern: unknown) => (u: any) => u.packexpansion((r: any) => r.pattern(X.DIALECT.spec(pattern)));

/** `(left op ... op right)`; a unary fold's other operand null. */
function fold(left: unknown, operator: string, right: unknown): any {
  return (u: any) => u.foldexpression((r: any) => {
    let found = r.operator(operator);
    if (left !== null) found = found.left(X.DIALECT.spec(left));
    return right === null ? found : found.right(X.DIALECT.spec(right));
  });
}

const throw_ = (operand: unknown = null) => (u: any) => u.throwexpression((r: any) => operand === null ? r : r.operand(X.DIALECT.spec(operand)));

<a id="CCS-01"></a>
## CCS-01 · Literals written back as each language writes them: a suffix by type, a typedef's value as the type it is, a floating value a binary64 literal spells; a value of a type with no literal a cast of a literal, its promoted arithmetic folding

In [ ]:
store.transaction(() => {
  // A value folds into the literal of its type, its suffix as C writes it; a typedef's value as the standard type it is
  for (const [spec, written] of [[C.binary("+", C.literal(1n, "u"), 1n), "2u"], [C.binary("*", C.literal(2n, "l"), 3n), "6l"],
    [C.binary("+", C.literal(1n, "ull"), 1n), "2ull"], [C.binary("<", 1n, 2n), "true"],
    [C.binary("+", C.cast(named("int64_t"), 5n), 1n), "6l"], [C.cast(named("size_t"), 7n), "7ul"],
    [C.binary("*", C.literal(1.5, "f"), 2n), "3.0f"], [C.binary("*", C.literal(1.5, "L"), 2n), "3.0L"],
    [C.binary("+", C.literal(0.1, "dd"), C.literal(0.2, "dd")), "0.3dd"],
    [C.binary("*", C.literal(0.5, "f16"), 3n), "1.5f16"], [string("hi"), '"hi"'], [nullptr, "nullptr"]] as [unknown, string][]) {
    assert(text(spec) === written && simplified(spec).literal, `${written}: ${text(spec)}`);
  }
  assert(value(C.binary("+", C.literal(1n, "u"), 1n)) === 2n && value(C.binary("*", C.literal(1.5, "L"), 2n)) === "3"); // natives
  // A floating value no binary64 literal spells exactly stays, as does a non-finite one; its value is the rules'
  for (const [abi, held] of [[amd64(), "0.33333333333333333334"], [aapcs64(), "0.3333333333333333333333333333333333"]] as [unknown, string][]) {
    assert(text(C.binary("/", C.literal(1.0, "L"), 3n), {}, C, abi) === "1.0L / 3" && rule(C.binary("/", C.literal(1.0, "L"), 3n), {}, C, abi).value === held);
  }
  assert(text(C.binary("/", 1.0, 0.0)) === "1.0 / 0.0" && rule(C.binary("/", 1.0, 0.0)).value === Infinity);
  // A value of a type with no literal is a cast of a literal that converts to it, a closed constructor expression: a char,
  // a short, a uint8_t's as the unsigned char it is; each ABI's char its own
  assert(text(C.cast(keywords("char"), C.binary("+", 100n, 100n))) === "(char)-56" && text(C.cast(keywords("char"), 200n), {}, C, aapcs64()) === "(char)200");
  assert(simplified(C.cast(keywords("char"), 200n)).constructed && !simplified(C.cast(keywords("char"), 200n)).literal);
  assert(value(C.cast(keywords("char"), 200n)) === -56n && rule(C.cast(keywords("char"), 200n)).value === -56n);
  assert(text(C.cast(keywords("short"), C.binary("+", 1n, 2n))) === "(short)3" && text(C.cast(named("uint8_t"), 300n)) === "(unsigned char)44");
  assert(text(C.cast(keywords("unsigned", "short"), 7n)) === "(unsigned short)7" && text(C.cast(named("int8_t"), 200n)) === "(signed char)-56");
  assert(text(C.binary("+", x, C.cast(keywords("char"), 200n))) === "x + (char)-56"); // a subterm, where the root stays
  // Its arithmetic, promoted to int, folds, as each ABI says: char is signed under AMD64, unsigned under AAPCS64
  assert(text(C.binary("+", C.cast(keywords("char"), 200n), 0n)) === "-56" && text(C.binary("+", C.cast(keywords("char"), 200n), 0n), {}, C, aapcs64()) === "200");
  // A typedef of an integer type is that type in C; C++'s character types are types of their own
  assert(text(C.cast(named("wchar_t"), 65n)) === "65" && text(C.cast(named("wchar_t"), 65n), {}, C, aapcs64()) === "65u");
  assert(text(X.cast(named("wchar_t"), 65n), {}, X) === "(wchar_t)65" && text(X.cast(named("int64_t"), 65n), {}, X) === "65l");
  assert(text(X.cast(named("char16_t"), 65n), {}, X) === "(char16_t)65" && text(X.cast(keywords("char"), 65n), {}, X) === "(char)65");
  // C++ writes its own extended floating types' suffixes
  assert(text(X.binary("*", X.literal(1.5, "f16"), 2n), {}, X) === "3.0f16" && text(X.binary("*", X.literal(1.5, "bf16"), 2n), {}, X) === "3.0bf16");
});

<a id="CCS-02"></a>
## CCS-02 · Parentheses, `__extension__`, adjacent strings, the comma and a cast to `void`

In [ ]:
store.transaction(() => {
  // Parentheses and GNU's `__extension__` are their operand; where it stays, the printer keeps what precedence needs
  assert(text(paren(C.binary("+", 1n, 2n))) === "3" && text(C.binary("*", paren(C.binary("+", x, 2n)), 3n)) === "(x + 2) * 3");
  assert(text(extension(C.binary("+", 1n, 2n))) === "3" && text(paren(x), { x: 4n }) === "4");
  // Adjacent string literals are one string; a prefixed one stays, as a prefixed string literal does
  assert(text(concat("ab", "cd")) === '"abcd"' && value(concat("ab", string("cd"))) === "abcd");
  assert(equal(reasons(concat("a", string("b", "L"))), [["C literal of wchar_t string is not simplified", "a"]]));
  // The comma: its left operand evaluated, then its right's value; a known left operand leaves the right
  assert(text(C.binary(",", 1n, 2n)) === "2" && text(C.binary(",", 1n, x)) === "x" && text(C.binary(",", x, 1n)) === "x, 1");
  raises(ZeroDivisionError, () => value(C.binary(",", C.binary("/", 1n, 0n), 2n)), "integer division by zero");
  // A cast to void evaluates its operand, to no value: it stays, its operand simplified
  assert(text(C.cast(keywords("void"), C.binary("+", 1n, 2n))) === "(void)3" && rule(C.cast(keywords("void"), 1n)).value === CV.VOID);
  assert(text(C.binary(",", C.cast(keywords("void"), 1n), 2n)) === "2");
  raises(TypeError, () => value(C.binary("+", C.cast(keywords("void"), 1n), 1n)), "invalid operand to +: 'void'");
  raises(NotEvaluated, () => value(C.cast(keywords("void"), 1n)), "the root is not a closed constructor expression: its value has no closed constructor expression");
  assert(String(CV.VOID) === "void");
});

<a id="CCS-03"></a>
## CCS-03 · The conditional: the chosen operand in the two operands' type, C's promoted, C++'s one where they are one; GNU's `a ?: b`

In [ ]:
store.transaction(() => {
  // The chosen operand, converted to the type the usual arithmetic conversions give the two (ISO/IEC 9899, 6.5.15)
  assert(text(C.conditional(1n, 2n, 3.5)) === "2.0" && text(C.conditional(0n, 2n, C.literal(3n, "u"))) === "3u");
  assert(text(C.conditional(1n, C.literal(1n, "u"), -1n)) === "1u" && text(C.conditional(1n, 2n, C.literal(3n, "l"))) === "2l");
  // C promotes two bools to int; C++ keeps the one type both are
  assert(text(C.conditional(1n, true, false)) === "1" && text(X.conditional(1n, true, false), {}, X) === "true");
  // The other operand's type is read, not its value; where only its declaration gives it, the chosen one's is kept
  assert(text(C.conditional(1n, 2n, C.binary("/", 1.0, 0.0))) === "2.0" && text(C.conditional(1n, 2n, x), { x: 1.5 }) === "2.0");
  assert(text(C.conditional(1n, 2n, x)) === "2" && text(C.conditional(1n, 2n, C.member(x, "a")), { x: { a: 1.5 } }) === "2");
  assert(text(C.conditional(x, 1n, 2n)) === "x ? 1 : 2" && text(C.conditional(x, 1n, 2n), { x: 0n }) === "2");
  raises(TypeError, () => value(C.conditional(1n, C.literal(0.5, "df"), 1.0)),
    "invalid operands to conditional expression ('_Decimal32' and 'double')");
  // GNU's `a ?: b`: `a`, evaluated once, unless it is zero, then `b`; in their common type
  assert(text(gnu(5n, 7n)) === "5" && text(gnu(0n, 7n)) === "7" && text(gnu(0n, 7.5)) === "7.5" && text(gnu(2n, 7.5)) === "2.0");
  assert(text(gnu(x, 1n)) === "x ?: 1" && text(gnu(x, 1n), { x: 3n }) === "3");
  // An operand of no arithmetic type is chosen as it is
  assert(text(C.conditional(1n, string("a"), string("b"))) === '"a"' && text(C.conditional(1n, 2n, string("s"))) === "2");
  raises(TypeError, () => value(C.conditional(1n, 2n, C.binary("+", C.literal(0.1, "df"), 1.0))),
    "invalid operands to binary + ('_Decimal32' and 'double')"); // the other's type is still C's
  // What C may never evaluate stays, though it would raise: a branch, by an unknown condition; `&&`'s right operand
  assert(text(C.conditional(x, 1n, C.binary("/", 1n, 0n))) === "x ? 1 : 1 / 0" && text(C.binary("&&", x, C.binary("/", 1n, 0n))) === "x && 1 / 0");
});

<a id="CCS-04"></a>
## CCS-04 · `nullptr`: false, equal to itself and to the literal 0, the operand of nothing else

In [ ]:
store.transaction(() => {
  // `nullptr`, C23's and C++'s, is a literal; a variable bound to null is one too
  for (const of of [C, X] as Dialect[]) {
    assert(text(nullptr, {}, of) === "nullptr" && value(nullptr, {}, of) === null && simplified(nullptr, {}, of).literal);
    assert(text(of.identifier("p"), { p: null }, of) === "nullptr");
    // It is false, equal to itself and to the integer literal 0, a null pointer constant
    for (const [spec, written] of [[of.binary("==", nullptr, nullptr), "true"], [of.binary("!=", nullptr, 0n), "false"],
      [of.binary("==", paren(0n, of), nullptr), "true"], [of.unary("!", nullptr), "true"],
      [of.conditional(nullptr, 1n, 2n), "2"], [of.binary("||", nullptr, 0n), "false"],
      [of.cast(keywords("bool"), nullptr), "false"]] as [unknown, string][]) {
      assert(text(spec, {}, of) === written, `${written}: ${text(spec, {}, of)}`);
    }
    // It is the operand of nothing else
    for (const [spec, message] of [[of.binary("+", nullptr, 1n), "invalid operand to +: 'nullptr_t'"],
      [of.binary("<", nullptr, nullptr), "invalid operands to binary < ('nullptr_t' and 'nullptr_t')"],
      [of.binary("==", nullptr, 1n), "invalid operands to binary == ('nullptr_t' and 'int')"],
      [of.cast(keywords("int"), nullptr), "invalid operand to a conversion: 'nullptr_t'"]] as [unknown, string][]) {
      raises(TypeError, () => value(spec, {}, of), message);
    }
  }
  raises(ValueError, () => value((u: any) => u.constant((r: any) => r.singleton("nil"))), "a constant needs the singleton nullptr"); // a malformed term
});

<a id="CCS-05"></a>
## CCS-05 · `sizeof` and `alignof`: a type's layout under its holder's ABI, an operand's type unevaluated, arrays and strings; what stays, by class

In [ ]:
store.transaction(() => {
  // A type's size and alignment are its holder's ABI's: a `size_t`, written as the `unsigned long` it is under both
  for (const [type, size] of [[keywords("char"), "1ul"], [keywords("_Bool"), "1ul"], [keywords("short"), "2ul"], [keywords("int"), "4ul"],
    [keywords("long", "long"), "8ul"], [keywords("double"), "8ul"], [named("wchar_t"), "4ul"],
    [named("_Float16"), "2ul"], [named("_Decimal128"), "16ul"]] as [unknown, string][]) {
    assert(text(sizeof(type)) === size && text(sizeof(type), {}, C, aapcs64()) === size, size);
  }
  assert(text(sizeof(keywords("long", "double"))) === "16ul" && text(alignof(keywords("long", "double"))) === "16ul"); // x87's 80 bits in 16
  assert(rule(sizeof(keywords("long", "double")), {}, C, aapcs64()).value === 16n); // binary128
  // An array's size is its elements', its alignment theirs
  assert(text(sizeof(keywords("int", array(3n)))) === "12ul" && text(sizeof(keywords("int", array(3n, array(2n))))) === "24ul");
  assert(text(alignof(keywords("double", array(4n)))) === "8ul" && text(alignof(keywords("char", array()))) === "1ul");
  // A string literal is an array of its encoding's code units and a terminating zero
  assert(equal([string("abc"), string("é"), string("é", "u8"), string("ab", "u"), string("ab", "U"), string("ab", "L"),
    concat("ab", "cd")].map((s) => text(sizeof(s))), ["4ul", "3ul", "3ul", "6ul", "12ul", "12ul", "5ul"]));
  // An operand is not evaluated; its type is its kind's, C's comparisons an int, C++'s a bool
  assert(text(sizeof(C.binary("/", 1n, 0n))) === "4ul" && text(sizeof(C.binary("+", C.literal(1n, "l"), 1.5))) === "8ul");
  assert(text(sizeof(C.binary("<", 1n, 2n))) === "4ul" && text(sizeof(X.binary("<", 1n, 2n), X), {}, X) === "1ul");
  assert(text(sizeof(C.cast(keywords("char"), 1n))) === "1ul" && text(sizeof(C.unary("-", C.cast(keywords("char"), 1n)))) === "4ul"); // promoted
  assert(text(sizeof(C.conditional(x, 1n, 2.5))) === "8ul" && text(sizeof(C.binary(",", 1n, C.literal(1n, "u")))) === "4ul");
  // A variable's type is its value's; one the scope does not bind stays
  assert(text(sizeof(x), { x: 2n ** 40n }) === "8ul" && text(sizeof(x)) === "sizeof x");
  // What stays, by class: an operand whose declaration gives its type (c), a type that is not arithmetic (d), a layout
  // no ABI dialect stores (a)
  for (const [spec, reason] of [[sizeof(C.member(x, "a")), ["C sizeofexpression of an operand of unknown type is not simplified", "c"]],
    [alignof(C.call("f")), ["C alignofexpression of an operand of unknown type is not simplified", "c"]],
    [sizeof(keywords("int", pointer)), ["C sizeofexpression of a type that is not arithmetic is not simplified", "d"]],
    [sizeof(named("my_t")), ["C sizeofexpression of a type that is not arithmetic is not simplified", "d"]],
    [sizeof(C.literal(5n, "wb")), ["C sizeofexpression of a bit-precise integer type is not simplified", "a"]],
    [sizeof(nullptr), ["C sizeofexpression of nullptr_t is not simplified", "a"]]] as [unknown, string[]][]) {
    assert(equal(reasons(spec, { x: { a: 1n } })[0], reason), reason[0]);
  }
  // What C makes an error raises
  for (const [spec, message] of [[sizeof(keywords("void")), "invalid application of 'sizeof' to a void type"],
    [alignof(keywords("void")), "invalid application of '_Alignof' to a void type"],
    [sizeof(keywords("int", array())), "invalid application of 'sizeof' to an incomplete type"],
    [sizeof(keywords("int", array(0n))), "size of an array is not positive: 0"],
    [sizeof(keywords("int", array(1.5))), "size of an array has non-integer type 'double'"]] as [unknown, string][]) {
    raises(TypeError, () => value(spec), message);
  }
  // Every kind's type: parentheses, `sizeof`'s own, `_Generic`'s choice, `!`, a shift
  assert(equal([paren(1n), sizeof(keywords("int")), generic(1n, [keywords("int"), 1.5]), C.unary("!", 1n),
    C.binary("<<", C.literal(1n, "l"), 1n)].map((spec) => text(sizeof(spec))), ["4ul", "8ul", "8ul", "4ul", "8ul"]));
  assert(text(sizeof(X.unary("!", 1n), X), {}, X) === "1ul");
  // A conditional's: its operands' common type, `nullptr_t` of two `nullptr`s; an operand whose type is not known stays
  assert(equal(reasons(sizeof(C.conditional(x, nullptr, nullptr)))[0], ["C sizeofexpression of nullptr_t is not simplified", "a"]));
  for (const spec of [x, concat("a", x), C.unary("*", x), C.binary(",", 0n, string("abc")), C.cast(keywords("char", pointer), 0n),
    C.conditional(x, 1n, nullptr)]) {
    assert(equal(reasons(sizeof(spec), spec === x ? { x: "abc" } : {})[0], ["C sizeofexpression of an operand of unknown type is not simplified", "c"]));
  }
  assert(equal(reasons(sizeof(X.binary("<=>", 1n, 2n), X), {}, X)[0], ["Cpp sizeofexpression of an operand of unknown type is not simplified", "c"]));
  for (const [spec, message] of [[C.unary("-", nullptr), "invalid operand to -: 'nullptr_t'"], [C.unary("-", string("ab")), "invalid operand to -: 'array'"],
    [C.unary("-", C.cast(keywords("void"), 0n)), "invalid operand to -: 'void'"],
    [C.binary("+", C.literal(0.1, "df"), 1.0), "invalid operands to binary + ('_Decimal32' and 'double')"],
    [C.conditional(x, C.cast(keywords("void"), 0n), C.cast(keywords("void"), 1n)), "invalid application of 'sizeof' to a void type"]] as [unknown, string][]) {
    raises(TypeError, () => value(sizeof(spec)), message);
  }
  assert(equal(reasons(sizeof((u: any) => u.error((r: any) => r.text("1 +"))))[0], ["C error is not simplified", "b"])); // invalid syntax
  assert(equal([concat("a", string("b", "L")), gnu(1n, 1.5), compound(keywords("int"), 5n)].map((spec) => text(sizeof(spec))), ["12ul", "8ul", "4ul"]));
  // An operand `sizeof` does not evaluate stays where the size does, though it would raise
  assert(text(sizeof(C.binary("+", C.member(x, "a"), C.binary("/", 1n, 0n)))) === "sizeof(x.a + 1 / 0)");
});

<a id="CCS-06"></a>
## CCS-06 · `_Generic`: the association of its controlling operand's type, by C's identity of types, unevaluated; `default`; what stays, by class

In [ ]:
store.transaction(() => {
  // The association whose type is its controlling operand's, by C's identity of types, else `default`
  const [int_, long_, long_long] = [keywords("int"), keywords("long"), keywords("long", "long")];
  assert(text(generic(1n, [long_, 1n], [int_, 2n], [null, 3n])) === "2" && text(generic(1.5, [keywords("float"), 1n], [null, 9n])) === "9");
  assert(text(generic(C.literal(1n, "l"), [long_long, 1n], [long_, 2n])) === "2"); // long and long long are types of their own
  assert(text(generic(C.cast(named("int64_t"), 1n), [long_long, 1n], [long_, 2n])) === "2"); // int64_t is long, under both ABIs
  assert(text(generic(C.cast(named("int64_t"), 1n), [long_long, 1n], [long_, 2n]), {}, C, aapcs64()) === "2");
  assert(text(generic(C.cast(keywords("char"), 1n), [keywords("signed", "char"), 1n], [keywords("unsigned", "char"), 2n],
    [keywords("char"), 3n])) === "3"); // char is neither signed char nor unsigned char
  assert(text(generic(1n, [keywords("const", "int"), 1n], [null, 2n])) === "2"); // a qualified type, which no operand's is
  assert(text(generic(1n, [keywords("int", pointer), 1n], [int_, 2n])) === "2"); // a pointer, which no operand's is
  assert(text(generic(C.binary("/", 1n, 0n), [int_, 7n])) === "7"); // its operand is not evaluated
  assert(text(generic(keywords("double"), [keywords("double"), 1n], [null, 2n])) === "1"); // a type name as its operand (C2y)
  assert(text(generic(x, [int_, C.binary("+", x, 1n)], [keywords("double"), 0n]), { x: 41n }) === "42"); // by its value's type
  assert(text(generic(x, [int_, C.binary("+", x, 1n)])) === "_Generic(x, int: x + 1)");
  // What stays, by class: an operand whose declaration gives its type (c), an association of a type a declaration
  // names (d), which may be the operand's
  assert(equal(reasons(generic(C.member(x, "a"), [int_, 1n]), { x: { a: 1n } })[0], ["C genericselection of an operand of unknown type is not simplified", "c"]));
  assert(equal(reasons(generic(1n, [named("my_t"), 1n], [int_, 2n]))[0], ["C genericselection of an association of a type a declaration names is not simplified", "d"]));
  // What C makes an error raises
  for (const [spec, message] of [[generic(1.5, [int_, 1n]), "'_Generic' selector of type 'double' is not compatible with any association"],
    [generic(1n, [int_, 1n], [named("int32_t"), 2n]), "'_Generic' specifies two associations of type 'int'"]] as [unknown, string][]) {
    raises(TypeError, () => value(spec), message);
  }
  // `nullptr` and `void` are types of their own; a type name that is not arithmetic, a string (a pointer once converted)
  assert(text(generic(nullptr, [int_, 1n], [null, 2n])) === "2" && text(generic(C.cast(keywords("void"), 0n), [null, 3n])) === "3");
  assert(equal(reasons(generic(keywords("int", pointer), [null, 3n]))[0], ["C genericselection of a type that is not arithmetic is not simplified", "d"]));
  assert(equal(reasons(generic(string("abc"), [null, 3n]))[0], ["C genericselection of an operand of unknown type is not simplified", "c"]));
});

<a id="CCS-07"></a>
## CCS-07 · Compound literals: a scalar's value, an array's elements by position and designator, braces elided or written, an array of arrays, a character array's string; an array written as a compound literal

In [ ]:
let ints: any, chars: any;
store.transaction(() => {
  // A compound literal of an arithmetic type is its initializer's value, converted; an empty one its zero
  assert(text(compound(keywords("int"), 7n)) === "7" && text(compound(keywords("double"), 7n)) === "7.0" && text(compound(keywords("int"))) === "0");
  // Of an array, an array, written as a compound literal of its elements' literals; a subscript reads its elements, in
  // its element type
  ints = keywords("int", array());
  assert(text(compound(ints, 10n, C.binary("+", 1n, 2n))) === "(int[]){10, 3}" && text(C.subscript(compound(ints, 10n, 20n, 30n), 1n)) === "20");
  assert(equal(value(compound(ints, 10n, C.binary("+", 1n, 2n))), [10n, 3n]) && simplified(compound(ints, 10n)).constructed);
  assert(text(compound(keywords("short", array(3n)), 1n, designated(2n, 9n))) === "(short[]){1, 0, 9}");
  assert(text(compound(keywords("unsigned", array()), 1n)) === "(unsigned int[]){1u}");
  assert(text(C.subscript(compound(keywords("double", array()), 1n), 0n)) === "1.0");
  // Its length is its type's, else its initializer's; an element no item initializes is zero; a designator places one
  assert(text(C.subscript(compound(keywords("int", array(4n)), designated(2n, 9n)), 2n)) === "9");
  assert(text(C.subscript(compound(keywords("int", array(4n)), designated(2n, 9n)), 0n)) === "0");
  assert(text(C.subscript(compound(ints, 1n, designated(3n, 7n, 5n)), 4n)) === "7" && text(sizeof(compound(ints, 1n, designated(3n, 7n, 5n)))) === "24ul");
  assert(text(sizeof(compound(ints, 1n, 2n, 3n))) === "12ul" && text(sizeof(compound(keywords("short", array(5n))))) === "10ul");
  // A character array's string: its code units and its terminating zero
  chars = keywords("char", array());
  assert(text(C.cast(keywords("int"), C.subscript(compound(chars, "hi"), 1n))) === "105" && text(sizeof(compound(chars, "hi"))) === "3ul");
  assert(text(C.cast(keywords("int"), C.subscript(compound(keywords("char", array(2n)), "hi"), 1n))) === "105"); // no room for its zero
  // What C makes an error raises
  for (const [spec, message] of [[compound(keywords("int", array(2n)), 1n, 2n, 3n), "excess elements in an array initializer"],
    [compound(keywords("int"), 1n, 2n), "excess elements in a scalar initializer"],
    [compound(keywords("char", array(1n)), "hi"), "initializer-string for an array is too long"],
    [compound(ints, designated(-1n, 1n)), "array index in an initializer is negative: -1"]] as [unknown, string][]) {
    raises(TypeError, () => value(spec), message);
  }
  // What stays, by class: a structure's members, which a declaration gives (d)
  const point = (u: any) => u.typeid((i: any) => i.specifiers([(s: any) => s.classspecifier((c: any) => c.key("struct").name(
    (n: any) => n.identifier((d: any) => d.spelling("point"))).body(true))]));
  assert(equal(reasons(compound(point, 1n, 2n))[0], ["C compoundliteralexpression of a type that is not arithmetic is not simplified", "d"]));
  assert(reasons(compound(keywords("int"), compound(keywords("int"), 1n))).length === 0); // a compound literal is an item's value
  // A character array of one integer is initialized as any array; a designator is an index, its value not braced
  assert(text(C.cast(keywords("int"), C.subscript(compound(chars, 65n), 0n))) === "65");
  const field = (u: any) => u.designatedinitializer((r: any) => r.designators([(d: any) => d.fielddesignator(
    (f: any) => f.name((i: any) => i.spelling("a")))]).initializer((i: any) => i.equalinitializer((e: any) => e.value(C.literal(1n)))));
  const braced = (u: any) => u.designatedinitializer((r: any) => r.designators([(d: any) => d.indexdesignator(
    (i: any) => i.index(C.literal(0n)))]).initializer((i: any) => i.initializerlist((l: any) => l.items([C.literal(1n)]))));
  for (const [spec, message] of [[compound(ints, field), "an array's element is designated by its index"],
    [compound(ints, designated(1.5 as any, 1n)), "array index in an initializer has non-integer type 'double'"],
    [concat("a", x), "invalid operand to a string concatenation: 'int'"]] as [unknown, string][]) {
    raises(TypeError, () => value(spec, { x: 1n }), message);
  }
  assert(text(compound(ints, braced)) === "(int[]){1}"); // a scalar's braces
  assert(text(sizeof(compound(keywords("int", array(4n))))) === "16ul" && text(C.subscript(compound(keywords("int", array(4n))), 3n)) === "0");
  // A call is given an array's elements' natives
  const added = (items: bigint[]) => items.reduce((a, b) => a + b, 0n);
  const total = new CV.Scope({ x: [1n, 2n] }, { functions: { f: added } });
  assert(value(C.call("f", compound(ints, 1n, 2n)), total) === 3n && value(C.call("f", x), total) === 3n && value(C.call("f", x), new CV.Scope({ x: [] }, { functions: { f: added } })) === 0n);
});

<a id="CCS-08"></a>
## CCS-08 · `_BitInt(N)`: literals of their narrowest type, arithmetic unpromoted and ranked below a standard type, casts, a value written back as its narrowest literal where that is of its type, else as a cast of a literal

In [ ]:
store.transaction(() => {
  // A `wb` literal is of the narrowest bit-precise integer type that holds it, signed (`_BitInt(N)`, N >= 2) or not
  assert(rule(C.literal(5n, "wb")).ctype.name() === "_BitInt(4)" && rule(C.literal(5n, "uwb")).ctype.name() === "unsigned _BitInt(3)");
  // Its arithmetic is unpromoted, in the operands' common type: 3wb + 4wb, a _BitInt(4) 7, whose narrowest literal 7wb is
  assert(text(C.binary("+", C.literal(3n, "wb"), C.literal(4n, "wb"))) === "7wb");
  // A value whose narrowest literal is of another type is a cast of a literal: 7wb - 6wb is a _BitInt(4) 1, where 1wb is
  // a _BitInt(2); a value no standard type holds, a cast of its narrowest literal
  assert(text(C.binary("-", C.literal(7n, "wb"), C.literal(6n, "wb"))) === "(_BitInt(4))1");
  assert(text(C.binary("<<", C.cast(bits(70n), 1n), 65n)) === "(_BitInt(70))36893488147419103232wb");
  assert(equal([rule(C.binary("-", C.literal(7n, "wb"), C.literal(6n, "wb"))).ctype.name(), rule(C.binary("-", C.literal(7n, "wb"), C.literal(6n, "wb"))).value], ["_BitInt(4)", 1n]));
  // It ranks below a standard type of its width, so an int operand makes it an int; unsigned arithmetic wraps
  assert(text(C.binary("+", C.literal(1n, "wb"), 1n)) === "2" && text(C.binary("+", C.literal(7n, "uwb"), C.literal(1n, "uwb"))) === "(unsigned _BitInt(3))0");
  assert(rule(C.binary("+", C.literal(7n, "uwb"), C.literal(1n, "uwb"))).value === 0n);
  raises(OverflowError, () => value(C.binary("+", C.literal(1n, "wb"), C.literal(1n, "wb"))), "signed overflow in +: 2 does not fit '_BitInt(2)'");
  // A cast converts to it, as to any integer type, alike under both ABIs
  assert(text(C.cast(bits(7n), 200n)) === "-56wb" && text(C.cast(bits(7n), 200n), {}, C, aapcs64()) === "-56wb");
  assert(text(C.cast(bits(70n, true), -1n)) === `${2n ** 70n - 1n}uwb` && text(C.cast(bits(8n), 1n)) === "(_BitInt(8))1");
  raises(TypeError, () => value(C.cast(bits(1n), 0n)), "width of '_BitInt' is too small: 1");
  raises(TypeError, () => value(C.cast((u: any) => u.typeid((i: any) => i.specifiers([(s: any) => s.primitivetypespecifier((p: any) => p.keyword("unsigned")),
    (s: any) => s.bitintspecifier((b: any) => b.width(C.literal(1.5)))])), 0n)), "width of 'unsigned _BitInt' has non-integer type 'double'");
  // An unsigned bit-precise type of a standard signed type's width ranks below it: their common type is its unsigned one
  assert(text(C.binary("+", C.cast(bits(32n, true), 1n), 1n)) === "2u");
  raises(TypeError, () => value(C.cast((u: any) => u.typeid((i: any) => i.specifiers([(s: any) => s.primitivetypespecifier((p: any) => p.keyword("long")),
    (s: any) => s.bitintspecifier((b: any) => b.width(C.literal(8n)))])), 1n)), "invalid type specifiers with '_BitInt': long");
});

<a id="CCS-09"></a>
## CCS-09 · C++'s names: identifiers, qualified names by their spelling, `this`; user-defined literals by the scope's literal operators, of one literal or adjacent strings

In [ ]:
store.transaction(() => {
  // An identifier is the scope's variable of its name; a qualified name the variable of its spelling
  assert(text(X.binary("+", ident("x"), 1n), { x: 2n }, X) === "3" && text(qualified("ns", "x"), { "ns::x": 5n }, X) === "5");
  assert(text(qualified("std", "numbers", "pi"), { "std::numbers::pi": 3.14 }, X) === "3.14" && text(qualified("ns", "x"), {}, X) === "ns::x");
  // `this` is the scope's variable `this`, read through `->` as any object is
  assert(text(X.member(this_, "a", "->"), { this: { a: 3n } }, X) === "3" && text(this_, {}, X) === "this");
  // A name that is neither stays: a declaration's (d)
  const operator_plus = (u: any) => u.idexpression((r: any) => r.name((n: any) => n.operatorname((o: any) => o.operator("+"))));
  assert(equal(reasons(operator_plus, {}, X), [["Cpp idexpression of a name that is not an identifier is not simplified", "d"]]));
  // A user-defined literal calls the literal operator the scope provides, by its name, with its literal's value
  const km = new XV.Scope({}, { functions: { 'operator""_km': (metres: bigint) => metres * 1000n } });
  assert(text(udl(12n, "_km"), km, X) === "12000" && text(X.binary("+", udl(1n, "_km"), udl(2n, "_km")), km, X) === "3000");
  assert(text(udl(12n, "_km"), {}, X) === "12_km"); // one the scope does not provide stays
  // Adjacent strings with a suffix, `"ab" "c"_len`: the literal operator called with their concatenation
  const strings = (...parts: unknown[]) => (u: any) => u.userdefinedliteral((r: any) => r.concatenation(
    (c: any) => c.parts(parts.map((part) => X.DIALECT.spec(part)))).suffix("_len"));
  const length = new XV.Scope({ x: 1n }, { functions: { 'operator""_len': (held: string) => BigInt(held.length) } });
  assert(text(strings(string("ab"), string("c")), length, X) === "3" && text(strings(string("ab"), string("c")), {}, X) === '"ab" "c"_len');
  raises(TypeError, () => value(strings(string("a"), ident("x")), length, X), "invalid operand to a string concatenation: 'int'");
  // A qualified name whose qualifiers are not identifiers stays too
  const in_template = (u: any) => u.idexpression((r: any) => r.name((n: any) => n.qualifiedname((q: any) => q.qualifiers([ // v<int>::x
    (m: any) => m.templateid((t: any) => t.name((k: any) => k.identifier((i: any) => i.spelling("v"))).arguments([keywords("int")]))]).name(
    (m: any) => m.identifier((i: any) => i.spelling("x"))))));
  assert(equal(reasons(in_template, {}, X)[0], ["Cpp idexpression of a name that is not an identifier is not simplified", "d"]));
  const global_x = (u: any) => u.idexpression((r: any) => r.name((n: any) => n.qualifiedname( // ::x
    (q: any) => q.global_scope(true).name((m: any) => m.identifier((i: any) => i.spelling("x"))))));
  assert(text(global_x, { "::x": 7n }, X) === "7");
});

<a id="CCS-10"></a>
## CCS-10 · C++'s casts: functional casts, braced ones refusing a narrowing conversion, `static_cast`; the object model's casts stay

In [ ]:
store.transaction(() => {
  // A functional cast to an arithmetic type converts as a cast does; with no value, its zero
  assert(text(functional("int", [2.9]), {}, X) === "2" && text(functional("int"), {}, X) === "0" && text(functional("double", [2n]), {}, X) === "2.0");
  assert(text(functional("unsigned", [-1n]), {}, X) === "4294967295u" && text(functional("int", [2n], true), {}, X) === "2");
  assert(text(functional("double", [2n], true), {}, X) === "2.0" && text(functional("int", [], true), {}, X) === "0");
  // A braced one refuses a narrowing conversion: to bool, from a floating type to an integer one, or one that changes its value
  for (const [spec, message] of [[functional("int", [2.5], true), "narrowing conversion of 2.5 from 'double' to 'int'"],
    [functional("uint8_t", [300n], true), "narrowing conversion of 300 from 'int' to 'uint8_t'"],
    [functional("bool", [1n], true), "narrowing conversion of 1 from 'int' to 'bool'"],
    [functional("int", [1n, 2n]), "a functional cast to 'int' of 2 values"]] as [unknown, string][]) {
    raises(TypeError, () => value(spec, {}, X), message);
  }
  assert(text(functional("uint8_t", [200n], true), {}, X) === "(unsigned char)200"); // no literal: a cast of one, as C writes it
  // `static_cast` is a cast; to `void`, its operand evaluated to no value
  assert(text(static_cast(keywords("double"), 1n), {}, X) === "1.0" && text(static_cast(keywords("int"), 2.9), {}, X) === "2");
  assert(text(static_cast(keywords("void"), X.binary("+", 1n, 2n)), {}, X) === "static_cast<void>(3)");
  // What stays, by class: the object model's casts, and a cast to a class (d)
  assert(equal(reasons(static_cast(keywords("int"), 1n, "reinterpret_cast"), {}, X)[0], ["Cpp namedcastexpression 'reinterpret_cast' is not simplified", "d"]));
  assert(equal(reasons(functional("point", [1n]), {}, X)[0], ["Cpp functionalcastexpression to a type that is not arithmetic is not simplified", "d"]));
  assert(equal(reasons(static_cast(named("point"), 1n), {}, X)[0], ["Cpp namedcastexpression to a type that is not arithmetic is not simplified", "d"]));
  // `void()` is no value; a type a `typename` specifier names is a declaration's
  assert(text(functional("void"), {}, X) === "void()" && rule(functional("void"), {}, X).value === CV.VOID);
  const typename_t = (u: any) => u.functionalcastexpression((r: any) => r.type((s: any) => s.typenamespecifier(
    (n: any) => n.name((m: any) => m.identifier((i: any) => i.spelling("T"))))).initializer((i: any) => i.parenthesizedinitializer(
    (p: any) => p.arguments([X.literal(1n)]))));
  assert(equal(reasons(typename_t, {}, X)[0], ["Cpp functionalcastexpression to a type that is not arithmetic is not simplified", "d"]));
  // Each cast's type is its operand's type, unevaluated; a cast to a class's, or by the object model's casts, is not known
  assert(text(sizeof(functional("short", [2n]), X), {}, X) === "2ul" && text(sizeof(static_cast(keywords("char"), 1n), X), {}, X) === "1ul");
  for (const spec of [static_cast(keywords("int"), 1n, "reinterpret_cast"), functional("point", [1n])]) {
    assert(equal(reasons(sizeof(spec, X), {}, X)[0], ["Cpp sizeofexpression of an operand of unknown type is not simplified", "c"]));
  }
});

<a id="CCS-11"></a>
## CCS-11 · C++'s `<=>`: `<compare>`'s orderings, compared with the literal 0 and with one another, a narrowing conversion refused

In [ ]:
store.transaction(() => {
  // `<=>` of two integers is a strong ordering; of floating values a partial one, unordered where one is NaN
  const three = (a: unknown, b: unknown) => X.binary("<=>", a, b);
  assert(equal(([[1n, 2n], [2n, 2n], [3n, 2n], [1.0, 2n], [2.0, 2n], [1.0, NaN]] as [unknown, unknown][]).map((pair) => String(rule(three(...pair), {}, X).value)), [
    "std::strong_ordering::less", "std::strong_ordering::equal", "std::strong_ordering::greater",
    "std::partial_ordering::less", "std::partial_ordering::equivalent", "std::partial_ordering::unordered"]));
  assert(rule(three(1n, 2n), {}, X).value instanceof XV.Ordering && rule(three(1n, 2n), {}, X).value.order === -1);
  assert(text(three(1n, 2n), {}, X) === "1 <=> 2"); // an ordering has no literal
  // It compares with the literal 0, on either side, and with an ordering of its category by == and !=
  assert(equal(["<", "<=", ">", ">=", "==", "!="].map((op) => text(X.binary(op, three(1n, 2n), 0n), {}, X)), [
    "true", "true", "false", "false", "false", "true"]));
  assert(text(X.binary(">", 0n, three(1n, 2n)), {}, X) === "true" && String(rule(three(0n, three(1n, 2n)), {}, X).value) === "std::strong_ordering::greater");
  assert(equal(["<", "==", "!="].map((op) => text(X.binary(op, three(1.0, NaN), 0n), {}, X)), ["false", "false", "true"]));
  assert(text(X.binary("==", three(2n, 2n), three(3n, 3n)), {}, X) === "true" && text(X.binary("!=", three(1n, 2n), three(2n, 1n)), {}, X) === "true");
  // What C++ makes an error raises: a narrowing conversion of an integer, an ordering compared with anything but 0
  for (const [spec, message] of [[three(-1n, X.literal(0n, "u")), "invalid operands to binary <=>: narrowing conversion of -1 from 'int' to 'unsigned int'"],
    [X.binary("<", three(1n, 2n), 1n), "invalid operands to binary < ('std::strong_ordering' and 'int')"],
    [X.binary("==", three(1n, 2n), three(1.0, 2n)), "invalid operands to binary == ('std::strong_ordering' and 'std::partial_ordering')"]] as [unknown, string][]) {
    raises(TypeError, () => value(spec, {}, X), message);
  }
  assert(String(rule(three(0n, three(1.0, NaN)), {}, X).value) === "std::partial_ordering::unordered");
  assert(String(rule(three(three(1n, 2n), 0n), {}, X).value) === "std::strong_ordering::less"); // an ordering <=> 0 is itself
});

<a id="CCS-12"></a>
## CCS-12 · C++'s packs: `sizeof...`, pack expansions in a call's arguments, unary and binary folds, empty folds; what stays

In [ ]:
store.transaction(() => {
  // A pack is the scope's: `sizeof...` its size, an expansion its values in a call's arguments, each by its pattern
  const [args, k] = [ident("args"), ident("k")];
  const sum = (...values: bigint[]) => values.reduce((a, b) => a + b, 0n);
  const scope = new XV.Scope({ k: 10n }, { functions: { f: sum }, packs: { args: [1n, 2n, 3n], none: [] } });
  assert(text(sizeof_pack("args"), scope, X) === "3ul" && text(sizeof_pack("rest"), scope, X) === "sizeof...(rest)");
  assert(text(X.call("f", expansion(args)), scope, X) === "6" && text(X.call("f", expansion(X.binary("*", args, k))), scope, X) === "60");
  assert(text(X.call("f", 100n, expansion(args)), scope, X) === "106");
  // A fold: right, `(args op ...)`, `(args op ... op i)`; left, `(... op args)`, `(i op ... op args)`
  assert(equal([fold(args, "+", null), fold(null, "-", args), fold(args, "-", 10n), fold(10n, "-", args), fold(args, "*", null)]
    .map((spec) => text(spec, scope, X)), ["6", "-4", "-8", "4", "6"]));
  // An empty fold: `&&` true, `||` false, `,` no value
  assert(text(fold(ident("none"), "&&", null), scope, X) === "true" && text(fold(null, "||", ident("none")), scope, X) === "false");
  assert(rule(fold(ident("none"), ",", null), scope, X).value === CV.VOID);
  // A pack the scope does not provide stays, named among the free names
  assert(text(fold(ident("rest"), "+", null), scope, X) === "(rest + ...)" && equal([...simplified(fold(ident("rest"), "+", null), scope, X).free], ["rest"]));
  // What C++ makes an error raises
  const two = new XV.Scope({}, { functions: { f: sum }, packs: { a: [1n], b: [1n, 2n] } });
  for (const [spec, within, message] of [[fold(ident("none"), "+", null), scope, "a fold of an empty pack over '+'"],
    [X.call("f", expansion(X.binary("+", ident("a"), ident("b")))), two, "the packs a pattern expands are of different lengths"],
    [X.call("f", expansion(k)), scope, "a pack expansion's pattern names no parameter pack"]] as [unknown, unknown, string][]) {
    raises(TypeError, () => value(spec, within, X), message);
  }
  // A fold over an assignment stays: a write (d)
  assert(equal(reasons(fold(args, "=", null), scope, X), [["Cpp foldexpression '=' is not simplified", "d"]]));
  raises(TypeError, () => value(fold(ident("ps"), "==", null), new XV.Scope({}, { packs: { ps: [null, 1n] } }), X),
    "invalid operands to binary == ('nullptr_t' and 'int')"); // a fold's operands are no literal 0
});

<a id="CCS-13"></a>
## CCS-13 · C++'s `throw`: thrown where it is evaluated, carrying its value; a rethrow stays

In [ ]:
store.transaction(() => {
  // `throw x` evaluates `x` and throws it, where it is evaluated; a conditional evaluates only the operand it chooses
  const c = ident("c");
  assert(text(X.conditional(c, 1n, throw_(7n)), { c: true }, X) === "1");
  raises(XV.Thrown, () => value(X.conditional(c, 1n, throw_(7n)), { c: false }, X), "an exception is thrown");
  try {
    value(throw_(X.binary("*", 6n, 7n)), {}, X);
    throw new Error("AssertionError: not thrown");
  } catch (thrown) {
    assert(thrown instanceof XV.Thrown && thrown.value === 42n); // it carries its operand's value
  }
  // A rethrow, `throw` alone, stays: a handler's (d)
  assert(equal(reasons(throw_(), {}, X), [["Cpp throwexpression without an operand is not simplified", "d"]]));
  assert(text(X.conditional(c, 1n, throw_(7n)), {}, X) === "c ? 1 : throw 7"); // by an unknown condition, it stays
});

<a id="CCS-14"></a>
## CCS-14 · What stays in the residual, by class: `FOLDED`, and what each dialect of the family leaves, named

In [ ]:
store.transaction(() => {
  // The kinds each dialect folds: every kind of class (a), and its literals and names
  assert(same([...CV.FOLDED].sort(), ["literal", "constant", "concatenatedstring", "variable", "parenthesizedexpression",
    "genericselection", "subscriptexpression", "callexpression", "memberexpression", "unaryexpression", "sizeofexpression",
    "alignofexpression", "castexpression", "binaryexpression", "conditionalexpression", "compoundliteralexpression",
    "extensionexpression"].sort()));
  assert(same([...XV.FOLDED].sort(), [...CV.FOLDED].filter((tag) => !["variable", "genericselection", "compoundliteralexpression"].includes(tag))
    .concat(["idexpression", "thisexpression", "userdefinedliteral", "functionalcastexpression", "namedcastexpression",
      "sizeofpackexpression", "foldexpression", "packexpansion", "throwexpression"]).sort()));
  // What stays, named with its class: the object model, writes and statements (d); an initializer alone, a type name (b)
  const assign = (u: any) => u.assignmentexpression((r: any) => r.left(x).operator("=").right(C.literal(1n)));
  for (const [spec, reason] of [[C.unary("*", x), ["C unaryexpression '*' is not simplified", "d"]],
    [C.unary("&", x), ["C unaryexpression '&' is not simplified", "d"]],
    [assign, ["C assignmentexpression is not simplified", "d"]],
    [(u: any) => u.statementexpression((r: any) => r), ["C statementexpression is not simplified", "d"]],
    [(u: any) => u.initializerlist((r: any) => r.items([C.literal(1n)])), ["C initializerlist is not simplified", "b"]],
    [keywords("int"), ["C Types typeid is not simplified", "b"]]] as [unknown, string[]][]) {
    assert(equal(reasons(spec, { x: 1n })[0], reason), reason[0]);
  }
  const dot_star = X.binary(".*", ident("o"), ident("m"));
  for (const [spec, reason] of [[(u: any) => u.newexpression((r: any) => r.type(keywords("int"))), ["Cpp newexpression is not simplified", "d"]],
    [(u: any) => u.typeidexpression((r: any) => r.operand(X.literal(1n))), ["Cpp typeidexpression is not simplified", "c"]],
    [dot_star, ["Cpp binaryexpression '.*' is not simplified", "d"]],
    [static_cast(keywords("int"), 1n, "const_cast"), ["Cpp namedcastexpression 'const_cast' is not simplified", "d"]]] as [unknown, string[]][]) {
    assert(equal(reasons(spec, {}, X)[0], reason), reason[0]);
  }
  assert(CV.CATEGORIES["C"]?.["sizeofexpression of an operand of unknown type"] === "c" && XV.CATEGORIES["Cpp"]?.["foldexpression '+='"] === "d");
  // Inference of a cast to a type that is not arithmetic: not known
  assert(C.DIALECT.infer(term(C.cast(keywords("char", pointer), 0n))) === CDialect.Domains.Anything);
  const enumerator = (u: any) => u.enumerator((e: any) => e.name((i: any) => i.spelling("A"))); // a type name holds none
  assert(equal(reasons(C.cast(enumerator, 1n))[0], ["C castexpression to a type that is not arithmetic is not simplified", "d"]));
});

<a id="CCS-15"></a>
## CCS-15 · Arrays of arrays and nested braces, initialized as C initializes them, each written as a compound literal; what has no closed constructor expression stays

In [ ]:
store.transaction(() => {
  // An array of arrays: its items fill its scalars in order, their braces elided, or each subarray its braces' items
  const pairs = keywords("int", array(2n, array())); // int[][2]
  const braces = (...items: unknown[]) => (u: any) => u.initializerlist((r: any) => r.items(items.map((item) => C.DIALECT.spec(item))));
  assert(text(compound(pairs, 1n, 2n, 3n)) === "(int[][2]){{1, 2}, {3, 0}}" && equal(value(compound(pairs, 1n, 2n, 3n)), [[1n, 2n], [3n, 0n]]));
  assert(text(compound(pairs, braces(1n), 5n)) === "(int[][2]){{1, 0}, {5, 0}}" && text(compound(pairs, 1n, braces(2n), 3n)) === "(int[][2]){{1, 2}, {3, 0}}");
  assert(text(sizeof(compound(pairs, 1n, 2n, 3n))) === "16ul" && text(C.subscript(C.subscript(compound(pairs, 1n, 2n, 3n), 1n), 0n)) === "3");
  // A scalar's braces, in an array; `{}` its zero
  assert(text(compound(ints, braces(1n), 2n)) === "(int[]){1, 2}" && text(compound(ints, braces(), 2n)) === "(int[]){0, 2}");

  /** `[i][j] = item`, an index a pair for GNU's range; `item` an array, its braces' items, or an expression. */
  function designation(indices: (bigint | [bigint, bigint])[], item: unknown): any {
    const designator = (index: bigint | [bigint, bigint]) => {
      const [first, last] = typeof index === "bigint" ? [index, null] : index;
      return (d: any) => d.indexdesignator((r: any) => last === null ? r.index(C.literal(first)) : r.index(C.literal(first)).last(C.literal(last)));
    };
    const initializer = Array.isArray(item) ? (i: any) => i.initializerlist((l: any) => l.items(item.map((v) => C.DIALECT.spec(v))))
      : (i: any) => i.equalinitializer((e: any) => e.value(C.DIALECT.spec(item)));
    return (u: any) => u.designatedinitializer((r: any) => r.designators(indices.map(designator)).initializer(initializer));
  }

  // A designation names an element, a subarray's among them, and the next item follows it; GNU's range repeats one; a
  // later item overrides what an earlier one initialized, braces all of their object
  assert(text(compound(pairs, designation([1n, 0n], 5n), 6n)) === "(int[][2]){{0, 0}, {5, 6}}");
  assert(text(compound(pairs, designation([1n], [7n, 8n]))) === "(int[][2]){{0, 0}, {7, 8}}");
  assert(text(compound(keywords("int", array(2n, array(2n))), designation([[0n, 1n]], [1n, 2n]))) === "(int[][2]){{1, 2}, {1, 2}}");
  assert(text(compound(pairs, designation([0n, 1n], 5n), designation([0n], [1n]))) === "(int[][2]){{1, 0}}");
  // A string initializes the first character array it reaches, in braces or not
  const chars3 = keywords("char", array(3n, array())); // char[][3]
  assert(text(compound(chars3, "ab", braces("cd"))) === "(char[][3]){{97, 98, 0}, {99, 100, 0}}");
  assert(text(compound(keywords("char", array(3n, array(2n, array()))), "ab", "cd")) === "(char[][2][3]){{{97, 98, 0}, {99, 100, 0}}}");
  assert(text(compound(chars, C.binary("+", 60n, 5n))) === "(char[]){65}"); // an expression, which is no string
  // What C makes an error raises
  for (const [spec, message] of [[compound(keywords("int", array(1n, array(1n))), 1n, 2n), "excess elements in an array initializer"],
    [compound(pairs, designation([0n, 2n], 1n)), "excess elements in an array initializer"],
    [compound(ints, designation([0n, 0n], 1n)), "designator in initializer for scalar type 'int'"],
    [compound(keywords("int"), designation([0n], 1n)), "designator in initializer for scalar type 'int'"],
    [compound(keywords("int"), braces(1n)), "too many braces around a scalar initializer"],
    [compound(ints), "an array of unknown size is initialized by an empty initializer"],
    [compound(keywords("int", array(null, array())), 1n), "array has incomplete element type"]] as [unknown, string][]) {
    raises(TypeError, () => value(spec), message);
  }
  // What has no closed constructor expression stays: an array of a value no literal converts to; a list a variable holds,
  // empty or not; C++'s list, which has no compound literal
  const third = C.binary("/", C.literal(1.0, "L"), 3n);
  assert(text(compound(keywords("long", "double", array()), third)) === "(long double[]){1.0L / 3}");
  assert(text(compound(keywords("long", "double", array(1n, array())), braces(third))) === "(long double[][1]){{1.0L / 3}}");
  assert(text(x, { x: [1n, 2n] }) === "x" && text(x, { x: [] }) === "x" && text(ident("x"), { x: [1n, 2n] }, X) === "x");
});

<a id="CCS-16"></a>
## CCS-16 · The scope's option `casts`: `"write"`, the default, writes a value no literal spells as a cast of a literal; `"stay"` leaves it in the residual; any other value is refused

In [ ]:
store.transaction(() => {
  // The scope's option `casts` (CAP-305): "write", the default, writes a value of a type no literal has as a cast of a
  // literal; "stay" leaves its subterm in the residual, simplified in turn and named by no residue, so that a root that
  // stays is no closed constructor expression
  const stay = new CV.Scope({}, { options: { casts: "stay" } });
  const char = C.cast(keywords("char"), 200n);
  assert(text(char) === "(char)-56" && text(char, new CV.Scope({}, { options: { casts: "write" } })) === "(char)-56" && value(char) === -56n);
  for (const [spec, written] of [[char, "(char)200"], [C.cast(keywords("short"), C.binary("+", 1n, 2n)), "(short)3"],
    [C.cast(named("uint8_t"), 300n), "(uint8_t)300"], [C.cast(bits(4n), 1n), "(_BitInt(4))1"],
    [C.binary("+", x, char), "x + (char)200"]] as [unknown, string][]) {
    const kept = simplified(spec, stay);
    assert(text(spec, stay) === written && !kept.constructed && kept.residues.length === 0, written);
  }
  raises(NotEvaluated, () => value(char, stay), "the root is not a closed constructor expression: its value has no closed constructor expression");
  // It changes only what a cast would write: a value with a literal folds as before, a cast's promoted arithmetic among
  // them, and C's compound literal of an array, whose elements are literals, is written
  for (const spec of [C.binary("+", char, 0n), C.cast(keywords("int"), 2.5), C.binary("*", C.literal(1.5, "f"), 2n), compound(chars, "hi")]) {
    assert(text(spec, stay) === text(spec) && simplified(spec, stay).constructed, text(spec));
  }
  // C++'s scope takes it too; a binding keeps it; the framework's option `errors` goes with it
  assert(text(X.cast(keywords("char"), 200n), new XV.Scope({}, { options: { casts: "stay" } }), X) === "(char)200");
  assert(stay.bind("y", 1n).option("casts", "write") === "stay");
  const both = new CV.Scope({}, { options: { casts: "stay", errors: "stay" } });
  assert(text(C.binary("+", char, C.binary("/", 1n, 0n)), both) === "(char)200 + 1 / 0");
  assert(equal(reasons(C.binary("+", char, C.binary("/", 1n, 0n)), both), [["C binaryexpression raising ZeroDivisionError is not simplified", "b"]]));
  // Any other value is refused, before anything is simplified
  raises(ValueError, () => simplified(C.binary("+", 1n, 2n), new CV.Scope({}, { options: { casts: "keep" } })), "the option 'casts' is 'write' or 'stay', not 'keep'");
});

<a id="CCS-17"></a>
## CCS-17 · A parameter assigned a value folds by C's rules, typed under the holder's ABI; one neither assigned nor bound stays where it stood, named, and C's short circuits fold it away

In [ ]:
store.transaction(() => {
  const n = store.References.OfParameter((p: any) => p.name("n"));
  const assigned = (held: unknown) => store.Contexts.OfContext((c: any) => c.parameters({
    n: (p: any) => p.name("n").value((v: any) => v.value(held)) }));
  // Assigned a value, a parameter folds by C's rules, its value typed under the holder's ABI as a variable's is
  assert(text(C.binary("+", n, 1n), assigned(2n)) === "3" && text(C.binary("*", n, 2n), assigned(2n ** 40n)) === "2199023255552l");
  assert(value(C.binary("+", C.identifier("n"), n), new CV.Scope({ n: 10n }).within(assigned(1n))) === 11n);
  // Neither assigned nor bound, it stays where it stood, named, while its sibling folds around it
  const kept = simplified(C.binary("+", n, C.binary("*", 2n, 3n)));
  assert([...kept.parameters].join() === "n" && F.same(term(kept.residual), term(C.binary("+", n, 6n))));
  // An ancestor folds it away where C's short circuit never evaluates it
  assert(text(C.binary("&&", 0n, n)) === "false" && text(C.binary("||", 1n, n)) === "true");
  assert([...simplified(C.binary("&&", 1n, n)).parameters].join() === "n");
  // Assigned a term, a parameter is read in its place where the place holds the term's kind, and folds; a term of a
  // kind its place does not hold, C++'s, stays, named
  assert(text(C.binary("*", n, 2n), assigned(term(C.binary("+", 1n, 2n)))) === "6");
  assert(equal(reasons(C.binary("*", n, 2n), assigned(term(X.binary("+", 1n, 2n), X))), [
    ["C parameter 'n' assigned a term its place does not hold is not simplified", "b"]]));
  const typeid = F.term(store.Expressions.Framework.Expression((e: any) => e.root(CTypes.Expressions.DIALECT.spec(keywords("int")))));
  assert(F.holds(store.registered("Expressions.C.Expressions.Term"), typeid)); // C's Term lists C Types' typeid
});

<a id="CCS-18"></a>
## CCS-18 · A type parameter assigned a type is read in its place: a cast's type and `sizeof`'s operand fold by it, a residual holds it where nothing folds, one neither assigned nor bound stays where it stood, named, and a term its place does not hold stays, named

In [ ]:
store.transaction(() => {
  const T = store.References.OfParameter((p: any) => p.name("T"));
  const typed = (spec: unknown) => F.term(store.Expressions.Framework.Expression((e: any) => e.root(CTypes.Expressions.DIALECT.spec(spec))));
  const assigned = (held: unknown) => store.Contexts.OfContext((c: any) => c.parameters({
    T: (p: any) => p.name("T").value((v: any) => v.value(held)) }));
  // A type parameter assigned a type is read in its place: a cast and `sizeof` fold by the type
  assert(text(C.cast(T, 3.75), assigned(typed(keywords("int")))) === "3");
  assert(text(C.cast(T, 300n), assigned(typed(keywords("unsigned", "char")))) === "(unsigned char)44");
  assert(text(sizeof(T), assigned(typed(keywords("double")))) === "8ul"); // an operand that may be a type or an expression
  // Where the type lets nothing fold, the residual holds it in the parameter's place
  assert(text(C.binary("+", C.cast(T, x), 1n), assigned(typed(keywords("int")))) === "(int)x + 1");
  // Neither assigned nor bound, it stays where it stood, named
  for (const spec of [C.cast(T, 3.75), sizeof(T)]) {
    const kept = simplified(spec);
    assert([...kept.parameters].join() === "T" && F.same(term(kept.residual), term(spec)));
  }
  // A term that its place does not hold, an expression where a type is expected, stays, named
  assert(equal(reasons(C.cast(T, 3.75), assigned(term(C.binary("+", 1n, 2n)))), [
    ["C parameter 'T' assigned a term its place does not hold is not simplified", "b"]]));
});